In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# 为了使绘图看起来更专业，我们使用一个内置的绘图风格
plt.style.use('seaborn-v0_8-whitegrid')

def generate_and_plot_dtft(a):
    """
    生成双边指数序列 x[n] = a^|n| 及其 DTFT。
    """
    
    # ---------------------------------------------------------
    # 1. 生成数据
    # ---------------------------------------------------------
    # 时域范围：生成从 -15 到 15 的整数序列
    n = np.arange(-15, 16)
    # 计算 x[n] = a^|n|
    # 注意：这里我们强制 abs(n) 为浮点数，以避免整数运算带来的问题
    x_n = a ** np.abs(n.astype(float))
    
    # 频域范围：生成从 -2*pi 到 2*pi 的频率轴
    omega = np.linspace(-2 * np.pi, 2 * np.pi, 1000)
    
    # 根据公式计算 DTFT: X(e^jω) = (1 - a^2) / (1 - 2a*cos(ω) + a^2)
    # 这个公式产生一个实值结果，我们在这里将其视为幅度
    with np.errstate(divide='ignore', invalid='ignore'):
        magnitude = (1 - a**2) / (1 - 2 * a * np.cos(omega) + a**2)
    
    # ---------------------------------------------------------
    # 2. 绘图设置
    # ---------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), constrained_layout=True)
    
    # --- 上图：时域信号 x[n] = a^|n| ---
    # 使用 stem 图。注意：已去掉了会导致报错的 use_line_collection 参数
    # 因为在现代 Matplotlib 中它是默认开启且不可调用的。
    markerline, stemlines, baseline = ax1.stem(n, x_n, linefmt='C0-', markerfmt='C0o', basefmt='k-')
    
    # 优化 stem 图的视觉效果
    plt.setp(markerline, markersize=6)
    plt.setp(stemlines, linewidth=1.5)
    
    # 设置标题和标签
    ax1.set_title(r'Time Domain: $x[n] = a^{|n|}$ (a = %.2f)' % a, fontsize=15)
    ax1.set_xlabel('n (Time Index)', fontsize=12)
    ax1.set_ylabel('Amplitude', fontsize=12)
    ax1.tick_params(axis='both', which='major', labelsize=10)
    
    # 动态调整 y 轴范围，以适应正负 'a' 值
    if a >= 0:
        ax1.set_ylim(-0.1, 1.1)
    else:
        # 当 a 为负，x[n] 震荡，计算其动态范围
        dynamic_range = np.max(np.abs(x_n))
        ax1.set_ylim(-dynamic_range * 1.1, dynamic_range * 1.1)
    
    # --- 下图：频域频谱 |X(e^jω)| ---
    ax2.plot(omega, magnitude, color='C1', linewidth=2.5)
    
    # 设置标题和标签
    ax2.set_title(r'Frequency Domain (DTFT): $X(e^{j\omega})$ (Magnitude Spectrum)', fontsize=15)
    ax2.set_ylabel('Amplitude', fontsize=12)
    
    # 格式化频域坐标轴
    ax2.axhline(0, color='black', linewidth=1)
    ax2.axvline(0, color='black', linewidth=1)
    
    # 设置 xticks 为 pi 的倍数，这对于理解 DTFT 周期性至关重要
    ax2.set_xticks([-2*np.pi, -np.pi, 0, np.pi, 2*np.pi])
    ax2.set_xticklabels([r'$-2\pi$', r'$-\pi$', '0', r'$\pi$', r'$2\pi$'], fontsize=12)
    ax2.set_xlabel(r'$\omega$ (Frequency in radians)', fontsize=12)
    ax2.tick_params(axis='y', labelsize=10)
    
    # 标注特殊点，例如 w=0 处的峰值
    # 值：(1+a)/(1-a)
    peak_val = (1 + a) / (1 - a)
    # 只有当峰值不在负无穷大或正无穷大时才标注（例如避免 a 接近 1 或 -1 时）
    if abs(peak_val) < 100:
        if a > 0:
            ax2.annotate(f'Peak: {peak_val:.2f}', xy=(0, peak_val), xytext=(np.pi/2, peak_val*0.9),
                         arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))
        elif a < 0:
            # 标注 w=pi 或 -pi 处的谷值/峰值
            valley_val = (1 - a) / (1 + a)
            ax2.annotate(f'Peak/Valley: {valley_val:.2f}', xy=(np.pi, valley_val), xytext=(np.pi*1.2, valley_val*1.2),
                         arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

    # 动态调整 y 轴范围，为标注和动态峰值留出空间
    y_max_plot = np.nanmax(np.append(magnitude, 1.0))
    ax2.set_ylim(-0.1 * y_max_plot, y_max_plot * 1.2)

    plt.show()

# ---------------------------------------------------------
# 3. 创建交互界面
# ---------------------------------------------------------
# 限制 'a' 的范围 (-0.95, 0.95)，因为由于公式分母，当 |a| >= 1 时系统不稳定且无法计算 DTFT
interact(generate_and_plot_dtft, a=FloatSlider(min=-0.95, max=0.95, step=0.05, value=0.5, description='a 值:', layout={'width': '800px'}));


interactive(children=(FloatSlider(value=0.5, description='a 值:', layout=Layout(width='800px'), max=0.95, min=-…